# Notebook 1: Acquire the Data

Download the January 2024 NYC TLC yellow and green cab trip records (Parquet format), the taxi zone shapefile and the zone lookup CSV from the NYC TLC public data archive. Once all files are on disk, verify row counts and file sizes using a standalone DuckDB connection.

## Prerequisites

The directory `~/neo4j-duckdb-taxi/data/raw` is created automatically if it doesn't exist. No environment variables are required for this notebook.

## 1. Install dependencies

In [1]:
%pip install duckdb==1.5.6 \
             pyproj==3.7.1 \
             pyshp==2.3.1 \
             requests==2.32.3 --quiet

Note: you may need to restart the kernel to use updated packages.


## 2. Imports and configuration

In [2]:
import duckdb
import json
import os
import requests
import shapefile
import zipfile

In [3]:
# Base data directory
RAW_DIR = os.path.expanduser("~/neo4j-duckdb-taxi/data/raw")
ZONES_DIR = os.path.join(RAW_DIR, "taxi_zones")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(ZONES_DIR, exist_ok=True)

# Download URLs
URLS = {
    "yellow_tripdata_2024-01.parquet": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet",
    "green_tripdata_2024-01.parquet":  "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2024-01.parquet",
    "taxi_zones.zip":                  "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip",
    "taxi+_zone_lookup.csv":           "https://d37ci6vzurychx.cloudfront.net/misc/taxi+_zone_lookup.csv",
}

## 3. Download raw files

In [4]:
def download_file(url, dest_path):
    if os.path.exists(dest_path):
        print(f"Already exists, skipping: {os.path.basename(dest_path)}")
        return
    print(f"Downloading {os.path.basename(dest_path)} ...")
    response = requests.get(url, stream=True)
    response.raise_for_status()
    total = int(response.headers.get("content-length", 0))
    downloaded = 0
    with open(dest_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                pct = downloaded / total * 100
                print(f"  {pct:.1f}%  ({downloaded / 1024 / 1024:.1f} MB)", end="\r")
    print(f"  Done: {downloaded / 1024 / 1024:.1f} MB")

In [5]:
for filename, url in URLS.items():
    dest = os.path.join(RAW_DIR, filename)
    download_file(url, dest)

  Done: 47.6 MB MB)
  Done: 1.3 MB MB)
  Done: 1.0 MB MB)
  Done: 0.0 MB


## 4. Extract taxi zones

In [6]:
zip_path = os.path.join(RAW_DIR, "taxi_zones.zip")
print("Extracting taxi_zones.zip ...")
with zipfile.ZipFile(zip_path, "r") as z:
    for member in z.namelist():
        # Strip the leading folder component so files land directly in ZONES_DIR
        filename = os.path.basename(member)
        if not filename:
            continue
        dest = os.path.join(ZONES_DIR, filename)
        with z.open(member) as src, open(dest, "wb") as dst:
            dst.write(src.read())
print(f"Extracted to: {ZONES_DIR.replace(os.path.expanduser('~'), '~')}")
print("Files:", sorted(os.listdir(ZONES_DIR)))

Extracting taxi_zones.zip ...
Extracted to: ~/neo4j-duckdb-taxi/data/raw/taxi_zones
Files: ['taxi_zones.cpg', 'taxi_zones.dbf', 'taxi_zones.prj', 'taxi_zones.shp', 'taxi_zones.shx']


## 5. Verify data

In [7]:
for filename in ["yellow_tripdata_2024-01.parquet", "green_tripdata_2024-01.parquet"]:
    path = os.path.join(RAW_DIR, filename)
    size_mb = os.path.getsize(path) / 1024 / 1024
    result = duckdb.sql(f"SELECT COUNT(*) AS row_count FROM read_parquet('{path}')").fetchone()
    print(f"{filename}")
    print(f"  Size:      {size_mb:.1f} MB")
    print(f"  Row count: {result[0]:,}")
    print()

yellow_tripdata_2024-01.parquet
  Size:      47.6 MB
  Row count: 2,964,624

green_tripdata_2024-01.parquet
  Size:      1.3 MB
  Row count: 56,551



In [8]:
csv_path = os.path.join(RAW_DIR, "taxi+_zone_lookup.csv")
print("Zone lookup CSV:")
result = duckdb.sql(f"SELECT COUNT(*) AS row_count FROM read_csv_auto('{csv_path}')").fetchone()
print(f"  Row count: {result[0]}")
print()
duckdb.sql(f"SELECT * FROM read_csv_auto('{csv_path}') LIMIT 5").show()

Zone lookup CSV:
  Row count: 265

┌────────────┬───────────────┬─────────────────────────┬──────────────┐
│ LocationID │    Borough    │          Zone           │ service_zone │
│   int64    │    varchar    │         varchar         │   varchar    │
├────────────┼───────────────┼─────────────────────────┼──────────────┤
│          1 │ EWR           │ Newark Airport          │ EWR          │
│          2 │ Queens        │ Jamaica Bay             │ Boro Zone    │
│          3 │ Bronx         │ Allerton/Pelham Gardens │ Boro Zone    │
│          4 │ Manhattan     │ Alphabet City           │ Yellow Zone  │
│          5 │ Staten Island │ Arden Heights           │ Boro Zone    │
└────────────┴───────────────┴─────────────────────────┴──────────────┘



## 6. Compute zone centroids

In [9]:
# Compute a lat/lon centroid for each taxi zone from the shapefile.
# pyshp reads the .shp; we average the polygon vertices for each zone
# and reproject from EPSG:2263 (NY State Plane feet) to WGS84 using pyproj.
#
# The output JSON is used by the Streamlit dashboard to place map markers.

from pyproj import Transformer

shp_path = os.path.join(ZONES_DIR, "taxi_zones.shp")
centroids_path = os.path.join(RAW_DIR, "zone_centroids.json")

transformer = Transformer.from_crs("EPSG:2263", "EPSG:4326", always_xy=True)

centroids = {}
with shapefile.Reader(shp_path) as sf:
    field_names = [f[0] for f in sf.fields[1:]]
    loc_idx = field_names.index("LocationID")

    for sr in sf.shapeRecords():
        location_id = sr.record[loc_idx]
        points = sr.shape.points
        if not points:
            continue
        x = sum(p[0] for p in points) / len(points)
        y = sum(p[1] for p in points) / len(points)
        lon, lat = transformer.transform(x, y)
        centroids[str(location_id)] = [round(lat, 6), round(lon, 6)]

with open(centroids_path, "w") as f:
    json.dump(centroids, f)

print(f"Zone centroids computed: {len(centroids)}")
print(f"Saved to: {centroids_path}")

for loc_id in ["1", "132", "161"]:
    print(f"  LocationID {loc_id}: {centroids[loc_id]}")

Zone centroids computed: 263
Saved to: /Users/akmalchaudhri/neo4j-duckdb-taxi/data/raw/zone_centroids.json
  LocationID 1: [40.690244, -74.174271]
  LocationID 132: [40.639071, -73.783741]
  LocationID 161: [40.757753, -73.97735]
